# 7.2 迁移学习与模型微调

本 notebook 介绍迁移学习的核心概念和两种主要的微调方法：冻结特征提取器和差异学习率

### 学习目标
- 理解迁移学习的概念：源域到目标域的知识迁移
- 掌握方法一：冻结特征提取器（requires_grad = False）
- 掌握方法二：差异学习率（参数组）
- 根据数据量和域相似度选择合适的微调策略

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import OrderedDict

print(f"PyTorch 版本: {torch.__version__}")

## 1. 迁移学习概念

**迁移学习**是将在**源域**（如 ImageNet）上学到的知识应用到**目标域**（如医学图像分类）的技术

### 为什么需要迁移学习？
- 目标域的标注数据通常很少
- 预训练模型已经学到了丰富的特征表示
- 底层特征（边缘、纹理）在不同任务间具有通用性
- 大幅减少训练时间和计算资源

### 迁移学习的流程
```
源域大数据集 -> 预训练模型 -> 目标域小数据集 -> 微调 -> 目标任务模型
(ImageNet)     (ResNet)     (医学图像)       (Fine-tune)  (疾病分类)
```

### 神经网络的层次特征
- **浅层**: 学习通用特征（边缘、颜色、纹理）
- **中层**: 学习中级特征（局部模式、部件）
- **深层**: 学习任务特定特征（物体类别）

## 2. 构建预训练模型（模拟）

为了演示微调过程，我们创建一个模拟的预训练模型。在实际项目中，通常使用 `torchvision.models` 提供的预训练模型

In [ ]:
class FeatureExtractor(nn.Module):
    """特征提取器（模拟预训练的卷积层）"""

    def __init__(self):
        super().__init__()
        # 模拟浅层特征：通用特征
        self.layer1 = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
        )
        # 模拟中层特征：中级特征
        self.layer2 = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
        )
        # 模拟深层特征：高级特征
        self.layer3 = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x


class PretrainedModel(nn.Module):
    """模拟预训练模型：特征提取器 + 分类头"""

    def __init__(self, num_classes: int = 100):
        super().__init__()
        self.features = FeatureExtractor()
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# 创建"预训练"模型（模拟在大数据集上训练过的模型）
pretrained_model = PretrainedModel(num_classes=100)
print("预训练模型结构:")
print(pretrained_model)

total_params = sum(p.numel() for p in pretrained_model.parameters())
print(f"\n总参数量: {total_params:,}")

In [ ]:
# 模拟"预训练"：在源域上训练几步
torch.manual_seed(42)
X_source = torch.randn(500, 20)
y_source = torch.randint(0, 100, (500,))

optimizer_pre = optim.Adam(pretrained_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

pretrained_model.train()
for step in range(50):
    output = pretrained_model(X_source)
    loss = criterion(output, y_source)
    optimizer_pre.zero_grad()
    loss.backward()
    optimizer_pre.step()

print(f"预训练完成, 最终损失: {loss.item():.4f}")

# 保存预训练权重
pretrained_state = pretrained_model.state_dict()
print(f"预训练权重 key: {list(pretrained_state.keys())}")

## 3. 准备目标域数据

目标域通常数据量较小，类别数也不同

In [ ]:
# 目标域：小数据集，5 个类别
torch.manual_seed(123)
X_target_train = torch.randn(80, 20)  # 只有 80 个训练样本
y_target_train = torch.randint(0, 5, (80,))

X_target_val = torch.randn(20, 20)
y_target_val = torch.randint(0, 5, (20,))

print(f"目标域训练集: {X_target_train.shape}, {y_target_train.shape}")
print(f"目标域验证集: {X_target_val.shape}, {y_target_val.shape}")
print(f"类别数: 5（源域为 100）")

## 4. 方法一：冻结特征提取器

**核心思想**: 保留预训练模型学到的特征，只训练新的分类头

**实现方式**: 将特征提取器的参数设置 `requires_grad = False`

**优点**:
- 训练速度快（只更新少量参数）
- 不容易过拟合（适合小数据集）
- 内存占用小

**适用场景**:
- 目标域数据量很少
- 目标域与源域相似

In [ ]:
def create_frozen_model(pretrained_state: dict, num_classes: int = 5) -> nn.Module:
    """创建冻结特征提取器的微调模型

    Args:
        pretrained_state: 预训练模型的 state_dict
        num_classes: 目标任务的类别数

    Returns:
        微调模型
    """
    # 步骤1: 创建模型并加载预训练权重
    model = PretrainedModel(num_classes=100)
    model.load_state_dict(pretrained_state)

    # 步骤2: 冻结特征提取器
    for param in model.features.parameters():
        param.requires_grad = False

    # 步骤3: 替换分类头（新的分类头默认 requires_grad=True）
    model.classifier = nn.Linear(64, num_classes)

    return model


# 创建微调模型
frozen_model = create_frozen_model(pretrained_state, num_classes=5)

# 查看哪些参数被冻结了
print("参数状态:")
for name, param in frozen_model.named_parameters():
    status = "可训练" if param.requires_grad else "已冻结"
    print(f"  {name}: {param.shape} -> {status}")

trainable = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in frozen_model.parameters())
print(f"\n可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

In [ ]:
# 重要：只将可训练参数传给优化器
# 这样可以节省内存，因为优化器不需要维护冻结参数的状态
optimizer_frozen = optim.Adam(
    filter(lambda p: p.requires_grad, frozen_model.parameters()),
    lr=0.001,
)

print(f"优化器管理的参数组数: {len(optimizer_frozen.param_groups)}")
print(f"优化器管理的参数数: {sum(len(g['params']) for g in optimizer_frozen.param_groups)}")

In [ ]:
# 训练冻结模型
criterion = nn.CrossEntropyLoss()

print("=== 方法一：冻结特征提取器 ===")
frozen_model.train()
for epoch in range(20):
    output = frozen_model(X_target_train)
    loss = criterion(output, y_target_train)
    optimizer_frozen.zero_grad()
    loss.backward()
    optimizer_frozen.step()

    if (epoch + 1) % 5 == 0:
        frozen_model.eval()
        with torch.no_grad():
            val_out = frozen_model(X_target_val)
            val_loss = criterion(val_out, y_target_val)
            val_acc = (val_out.argmax(1) == y_target_val).float().mean()
        print(
            f"Epoch {epoch+1:3d} | "
            f"train_loss={loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )
        frozen_model.train()

## 5. 方法二：差异学习率

**核心思想**: 不同层使用不同的学习率
- 浅层：使用很小的学习率（保护通用特征）
- 深层：使用较大的学习率（适配新任务）
- 分类头：使用最大的学习率（从零学习）

**实现方式**: 使用 PyTorch 优化器的**参数组（parameter groups）**功能

**优点**:
- 充分利用预训练特征
- 所有参数都能适应目标任务
- 效果通常更好

**适用场景**:
- 目标域有中等数据量
- 目标域与源域有一定差异

In [ ]:
def create_differential_lr_model(
    pretrained_state: dict,
    num_classes: int = 5,
    base_lr: float = 0.001,
) -> tuple:
    """创建使用差异学习率的微调模型

    Args:
        pretrained_state: 预训练模型的 state_dict
        num_classes: 目标任务的类别数
        base_lr: 基础学习率

    Returns:
        (模型, 优化器)
    """
    # 步骤1: 创建模型并加载预训练权重
    model = PretrainedModel(num_classes=100)
    model.load_state_dict(pretrained_state)

    # 步骤2: 替换分类头
    model.classifier = nn.Linear(64, num_classes)

    # 步骤3: 设置差异学习率的参数组
    param_groups = [
        # 浅层：最小学习率
        {
            'params': model.features.layer1.parameters(),
            'lr': base_lr * 0.01,   # 1/100 的基础学习率
            'name': 'layer1',
        },
        # 中层：中等学习率
        {
            'params': model.features.layer2.parameters(),
            'lr': base_lr * 0.1,    # 1/10 的基础学习率
            'name': 'layer2',
        },
        # 深层：较大学习率
        {
            'params': model.features.layer3.parameters(),
            'lr': base_lr * 0.5,    # 1/2 的基础学习率
            'name': 'layer3',
        },
        # 分类头：最大学习率
        {
            'params': model.classifier.parameters(),
            'lr': base_lr,          # 完整学习率
            'name': 'classifier',
        },
    ]

    optimizer = optim.Adam(param_groups)
    return model, optimizer


# 创建模型和优化器
diff_lr_model, optimizer_diff = create_differential_lr_model(
    pretrained_state,
    num_classes=5,
    base_lr=0.001,
)

# 查看每个参数组的学习率
print("参数组学习率:")
for group in optimizer_diff.param_groups:
    n_params = sum(p.numel() for p in group['params'])
    print(f"  {group['name']}: lr={group['lr']:.6f}, 参数数={n_params:,}")

In [ ]:
# 训练差异学习率模型
criterion = nn.CrossEntropyLoss()

print("=== 方法二：差异学习率 ===")
diff_lr_model.train()
for epoch in range(20):
    output = diff_lr_model(X_target_train)
    loss = criterion(output, y_target_train)
    optimizer_diff.zero_grad()
    loss.backward()
    optimizer_diff.step()

    if (epoch + 1) % 5 == 0:
        diff_lr_model.eval()
        with torch.no_grad():
            val_out = diff_lr_model(X_target_val)
            val_loss = criterion(val_out, y_target_val)
            val_acc = (val_out.argmax(1) == y_target_val).float().mean()
        print(
            f"Epoch {epoch+1:3d} | "
            f"train_loss={loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )
        diff_lr_model.train()

## 6. 如何选择微调策略

根据**目标域数据量**和**域相似度**来选择：

| | 目标域与源域相似 | 目标域与源域差异大 |
|---|---|---|
| **数据量少** | 冻结特征提取器，只训练分类头 | 冻结浅层，微调深层 + 分类头 |
| **数据量多** | 差异学习率，全网络微调 | 差异学习率，或从头训练 |

### 详细建议

**1. 少数据 + 高相似度**（如 ImageNet -> 自然图像分类）
- 冻结全部特征提取器
- 只训练分类头
- 防止过拟合

**2. 少数据 + 低相似度**（如 ImageNet -> 医学图像）
- 冻结浅层（通用特征可用）
- 微调深层 + 分类头
- 使用较小学习率

**3. 多数据 + 高相似度**（如 ImageNet -> 更大的自然图像数据集）
- 差异学习率
- 全网络微调
- 浅层小学习率，深层大学习率

**4. 多数据 + 低相似度**（如 ImageNet -> 卫星图像）
- 可以考虑从头训练
- 或使用差异学习率全网络微调
- 预训练权重仍可提供更好的初始化

## 7. 渐进式解冻（Progressive Unfreezing）

一种更高级的微调策略：从最后一层开始，逐步解冻更多层

In [ ]:
def progressive_unfreeze_demo():
    """渐进式解冻示例"""
    # 创建模型并加载预训练权重
    model = PretrainedModel(num_classes=100)
    model.load_state_dict(pretrained_state)
    model.classifier = nn.Linear(64, 5)

    # 初始状态：冻结所有特征层
    for param in model.features.parameters():
        param.requires_grad = False

    criterion = nn.CrossEntropyLoss()
    layers_to_unfreeze = [
        ('layer3', model.features.layer3),
        ('layer2', model.features.layer2),
        ('layer1', model.features.layer1),
    ]

    print("=== 渐进式解冻 ===")

    # 阶段0: 只训练分类头
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001,
    )
    print("\n阶段 0: 只训练分类头")
    model.train()
    for epoch in range(5):
        output = model(X_target_train)
        loss = criterion(output, y_target_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"  loss = {loss:.4f}")

    # 逐步解冻各层
    for i, (name, layer) in enumerate(layers_to_unfreeze):
        # 解冻当前层
        for param in layer.parameters():
            param.requires_grad = True

        # 重新创建优化器（包含新解冻的参数）
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=0.001 * (0.5 ** (i + 1)),  # 每次解冻降低学习率
        )

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"\n阶段 {i+1}: 解冻 {name} (可训练: {trainable}/{total})")

        model.train()
        for epoch in range(5):
            output = model(X_target_train)
            loss = criterion(output, y_target_train)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"  loss = {loss:.4f}, lr = {optimizer.param_groups[0]['lr']:.6f}")


progressive_unfreeze_demo()

## 8. 完整微调示例

将两种方法整合到一个完整的微调流程中

In [ ]:
def finetune(
    pretrained_state: dict,
    method: str = "freeze",
    num_classes: int = 5,
    epochs: int = 30,
    base_lr: float = 0.001,
) -> nn.Module:
    """完整的微调流程

    Args:
        pretrained_state: 预训练模型的 state_dict
        method: 微调方法, 'freeze' 或 'diff_lr'
        num_classes: 目标类别数
        epochs: 训练轮数
        base_lr: 基础学习率

    Returns:
        微调后的模型
    """
    # 构建模型
    model = PretrainedModel(num_classes=100)
    model.load_state_dict(pretrained_state)
    model.classifier = nn.Linear(64, num_classes)

    if method == "freeze":
        # 冻结特征提取器
        for param in model.features.parameters():
            param.requires_grad = False
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=base_lr,
        )
    elif method == "diff_lr":
        # 差异学习率
        optimizer = optim.Adam([
            {'params': model.features.layer1.parameters(), 'lr': base_lr * 0.01},
            {'params': model.features.layer2.parameters(), 'lr': base_lr * 0.1},
            {'params': model.features.layer3.parameters(), 'lr': base_lr * 0.5},
            {'params': model.classifier.parameters(), 'lr': base_lr},
        ])
    else:
        raise ValueError(f"未知方法: {method}")

    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"\n=== 微调方法: {method} ===")
    best_val_acc = 0.0

    for epoch in range(epochs):
        # 训练
        model.train()
        output = model(X_target_train)
        loss = criterion(output, y_target_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        # 验证
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_out = model(X_target_val)
                val_loss = criterion(val_out, y_target_val)
                val_acc = (val_out.argmax(1) == y_target_val).float().mean()
            best_val_acc = max(best_val_acc, val_acc.item())
            print(
                f"Epoch {epoch+1:3d} | "
                f"train_loss={loss:.4f} | "
                f"val_acc={val_acc:.4f}"
            )

    print(f"最佳验证准确率: {best_val_acc:.4f}")
    return model


# 对比两种方法
model_freeze = finetune(pretrained_state, method="freeze", epochs=30)
model_diff = finetune(pretrained_state, method="diff_lr", epochs=30)

## 9. 实际项目中的微调（torchvision 示例）

在实际项目中，通常使用 `torchvision.models` 中的预训练模型。以下是伪代码示例：

```python
import torchvision.models as models

# 加载预训练 ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 方法一：冻结特征提取器
for param in model.parameters():
    param.requires_grad = False

# 替换最后的全连接层
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

# 方法二：差异学习率
optimizer = optim.Adam([
    {'params': model.conv1.parameters(), 'lr': 1e-5},
    {'params': model.layer1.parameters(), 'lr': 1e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-4},
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-3},
    {'params': model.fc.parameters(), 'lr': 1e-2},
])
```

## 10. 总结

### 迁移学习核心概念
- 将源域知识迁移到目标域
- 浅层特征通用，深层特征任务相关
- 预训练模型提供好的初始化

### 两种微调方法

| 特性 | 冻结特征提取器 | 差异学习率 |
|------|--------------|----------|
| 训练参数 | 仅分类头 | 全部参数 |
| 训练速度 | 快 | 慢 |
| 过拟合风险 | 低 | 中等 |
| 适用数据量 | 少 | 中-多 |
| 实现方式 | `requires_grad=False` | 优化器参数组 |

### 最佳实践
1. 总是先尝试冻结特征提取器（简单、快速）
2. 如果效果不够好，再尝试差异学习率
3. 使用较小的学习率微调预训练层
4. 监控训练过程中的过拟合（训练/验证 loss 差距）
5. 考虑使用数据增强来减少过拟合

---

## 练习题

### 基础题
1. 创建一个预训练模型，分别使用冻结法和差异学习率法进行微调，对比验证准确率
2. 修改冻结策略：只冻结前两层，微调第三层和分类头

### 进阶题
3. 实现渐进式解冻：
   - 第 1-5 epoch: 只训练分类头
   - 第 6-10 epoch: 解冻最后一层
   - 第 11-15 epoch: 解冻所有层
   - 记录每个阶段的验证准确率变化

### 挑战题
4. 实现一个 `FineTuner` 类，支持：
   - 自动选择微调策略（基于数据量和验证集表现）
   - 支持冻结、差异学习率、渐进解冻三种方法
   - 提供统一的接口

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的更多练习题来巩固知识。